# Coalitions-01-Rust — Pouvoir coalitionnel : mêmes grandeurs, deux moteurs

**Navigation** : [<< Coalitions-01 (Python)](Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) | [Index de la sous-série](../README.md) | [Coalitions-02 (Python) >>](Coalitions-02-Assistance-Games-Python.ipynb)

**Jumeau Python** : [Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb](Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) — même sujet, même institution mathématique, implémentation indépendante en Rust.

Ce carnet est le **portage Rust** du jumeau Python. Il ne re-résout pas le sujet : il en donne une **seconde implémentation indépendante**, écrite avec le crate [`z3`](https://crates.io/crates/z3) et le solveur installé sur le poste. Les deux côtés calculent les mêmes grandeurs — Shapley–Shubik, Banzhaf absolu et normalisé, Deegan–Packel — par deux mécanismes distincts : ici, l'énumération combinatoire et l'énumération de modèles SMT.

Pourquoi cette redondance ? Parce qu'un résultat obtenu deux fois par deux chemins est plus solide qu'un résultat obtenu une fois : si une implémentation se trompe, l'autre le dit. La comparaison n'est pas une preuve d'égalité — elle porte sur un ensemble fini d'instances, comme le dit la section « Cross-validation » — mais c'est un **test différentiel** dont le détail est mesuré.

In [2]:
// Imports : le crate z3 (solveur SMT) et son API de modélisation.
// Le crate est lié dynamiquement à la bibliothèque Z3 installée sur le poste.
:dep z3 = { version = "0.23.0" }

use z3::ast::{Ast, Bool, Int};

println!("Imports OK : z3");
println!("z3 (lib) version : {}", z3::full_version());

Imports OK : z3


z3 (lib) version : 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e


## 1. Le jeu pondéré : poids, quota, coalitions

Un jeu de vote pondéré s'écrit $[q;w_1,\ldots,w_n]$. Une coalition gagne si la somme de ses poids atteint le quota $q$. Un joueur est **critique** (pivot) dans une coalition gagnante si son retrait la fait perdre. Une coalition gagnante est **minimale** si elle gagne mais qu'aucun de ses membres n'est superflu.

Le code ci-dessous définit le jeu et trois prédicats structurels : joueur **nul** (son entrée ne change jamais le résultat d'une coalition), joueur à **veto** (sans lui, même la coalition complète perd), **dictateur** (il gagne seul *et* il a le veto).

Le portage garde la sémantique du jumeau Python, y compris l'ordre d'énumération des coalitions : par taille croissante, puis ordre lexicographique. Cet ordre n'est pas décoratif — il rend les listes comparables terme à terme entre les deux côtés.

In [3]:
// Le jeu de vote pondéré : poids, quota, noms des joueurs.
// Même sémantique que le jumeau Python : une coalition gagne si la somme de
// ses poids atteint le quota ; un joueur est pivot si son retrait fait
// perdre une coalition gagnante.
struct VotingGame {
    weights: Vec<i64>,
    quota: i64,
    names: Vec<String>,
}

impl VotingGame {
    fn new(weights: Vec<i64>, quota: i64, names: Vec<&str>) -> VotingGame {
        VotingGame {
            weights,
            quota,
            names: names.iter().map(|s| s.to_string()).collect(),
        }
    }
    fn n(&self) -> usize {
        self.weights.len()
    }
    fn weight(&self, coalition: &[usize]) -> i64 {
        coalition.iter().map(|i| self.weights[*i]).sum()
    }
    fn wins(&self, coalition: &[usize]) -> bool {
        self.weight(coalition) >= self.quota
    }
    // Toutes les coalitions : par taille croissante, puis ordre lexicographique
    // (le même ordre que `itertools.combinations` côté Python).
    fn coalitions(&self) -> Vec<Vec<usize>> {
        let mut out: Vec<Vec<usize>> = Vec::new();
        for size in 0..=self.n() {
            let mut cur: Vec<usize> = Vec::new();
            gen_combos(self.n(), size, 0, &mut cur, &mut out);
        }
        out
    }
    // Les coalitions minimales gagnantes : gagnantes, et perdantes dès qu'on
    // retire l'un de leurs membres.
    fn minimal_winning(&self) -> Vec<Vec<usize>> {
        self.coalitions()
            .into_iter()
            .filter(|s| {
                self.wins(s)
                    && s.iter().all(|i| {
                        let without: Vec<usize> = s.iter().copied().filter(|j| j != i).collect();
                        !self.wins(&without)
                    })
            })
            .collect()
    }
    // Un joueur nul ne fait jamais basculer le résultat d'une coalition.
    fn is_dummy(&self, i: usize) -> bool {
        self.coalitions()
            .iter()
            .filter(|s| !s.contains(&i))
            .all(|s| {
                let mut with = s.clone();
                with.push(i);
                self.wins(&with) == self.wins(s)
            })
    }
    // Un joueur à veto est indispensable : sans lui, même la coalition
    // complète perd.
    fn is_veto(&self, i: usize) -> bool {
        let others: Vec<usize> = (0..self.n()).filter(|j| *j != i).collect();
        !self.wins(&others)
    }
    // Un dictateur gagne seul et dispose du veto.
    fn is_dictator(&self, i: usize) -> bool {
        self.wins(&[i]) && self.is_veto(i)
    }
}

// Combinaisons de `k` éléments parmi `0..n`, en ordre lexicographique.
fn gen_combos(n: usize, k: usize, start: usize, cur: &mut Vec<usize>, out: &mut Vec<Vec<usize>>) {
    if cur.len() == k {
        out.push(cur.clone());
        return;
    }
    let remaining = k - cur.len();
    for i in start..=(n - remaining) {
        cur.push(i);
        gen_combos(n, k, i + 1, cur, out);
        cur.pop();
    }
}

// Combinaisons de `k` éléments parmi une liste donnée, en ordre lexicographique.
fn combos_of(items: &[usize], k: usize) -> Vec<Vec<usize>> {
    let mut out: Vec<Vec<usize>> = Vec::new();
    fn rec(items: &[usize], k: usize, start: usize, cur: &mut Vec<usize>, out: &mut Vec<Vec<usize>>) {
        if cur.len() == k {
            out.push(cur.clone());
            return;
        }
        let remaining = k - cur.len();
        for idx in start..items.len() {
            if items.len() - idx < remaining {
                break;
            }
            cur.push(items[idx]);
            rec(items, k, idx + 1, cur, out);
            cur.pop();
        }
    }
    rec(items, k, 0, &mut Vec::new(), &mut out);
    out
}

// Les jeux de référence : l'exemple du début, un jeu symétrique, un jeu à
// quatre acteurs, et le jeu à joueur nul de l'exemple guidé.
fn game_exemple() -> VotingGame {
    VotingGame::new(vec![3, 2, 1], 4, vec!["A", "B", "C"])
}
fn game_symetrique() -> VotingGame {
    VotingGame::new(vec![1, 1, 1], 2, vec!["X", "Y", "Z"])
}
fn game_quatre() -> VotingGame {
    VotingGame::new(vec![5, 3, 2, 1], 6, vec!["P", "Q", "R", "S"])
}
fn game_joueur_nul() -> VotingGame {
    VotingGame::new(vec![5, 3, 2, 1], 10, vec!["A", "B", "C", "D"])
}

// Le jeu d'exemple : coalitions minimales et rôles structurels.
let game = game_exemple();
let coalitions_minimales: Vec<String> = game
    .minimal_winning()
    .iter()
    .map(|s| {
        let noms: Vec<String> = s.iter().map(|i| format!("'{}'", game.names[*i])).collect();
        format!("[{}]", noms.join(", "))
    })
    .collect();
println!("Coalitions gagnantes minimales : [{}]", coalitions_minimales.join(", "));
for i in 0..game.n() {
    println!(
        "{} : nul={} veto={} dictateur={}",
        game.names[i],
        game.is_dummy(i),
        game.is_veto(i),
        game.is_dictator(i)
    );
}
let _ = &game;

Coalitions gagnantes minimales : [['A', 'B'], ['A', 'C']]


A : nul=false veto=true dictateur=false


B : nul=false veto=false dictateur=false


C : nul=false veto=false dictateur=false


### Lecture — coalitions minimales et rôles structurels

Dans `[4; 3, 2, 1]`, les coalitions minimales gagnantes sont `{A,B}` et `{A,C}`. A n'est pas dictateur : son poids 3 reste sous le quota. Pourtant, A appartient à **toutes** les coalitions minimales ; cette structure annonce un pouvoir supérieur à sa part de poids.

Les rôles structurels concluent le diagnostic : aucun joueur nul, aucun dictateur, et un seul véto — A. L'arithmétique du véto est immédiate : sans A, B et C pèsent 2 + 1 = 3, sous le quota de 4 ; aucune coalition ne passe donc sans lui.

C'est le premier résultat que le solveur devra retrouver par une voie entièrement différente.

In [4]:
// Les indices de pouvoir, calculés par énumération explicite des coalitions :
// la voie combinatoire, sans solveur.

// Les coalitions où le joueur `i` est pivot — il ne fait pas gagner la
// coalition, mais il la fait gagner.
fn swing_counts(game: &VotingGame) -> Vec<i64> {
    let mut counts = vec![0i64; game.n()];
    for i in 0..game.n() {
        for s in game.coalitions() {
            if !s.contains(&i) {
                let mut with = s.clone();
                with.push(i);
                if !game.wins(&s) && game.wins(&with) {
                    counts[i] += 1;
                }
            }
        }
    }
    counts
}

fn factorial(n: usize) -> f64 {
    (1..=n).map(|k| k as f64).fold(1.0, |acc, k| acc * k)
}

// Les quatre grandeurs : Shapley-Shubik, Banzhaf absolu, Banzhaf normalisé,
// Deegan-Packel.
fn power_indices(game: &VotingGame) -> (Vec<f64>, Vec<f64>, Vec<f64>, Vec<f64>) {
    let n = game.n();
    // Shapley-Shubik : le joueur est pivot dans une coalition de taille s
    // avec le coefficient s! (n-s-1)! / n! (une coalition aléatoire).
    let mut shapley = vec![0.0f64; n];
    for i in 0..n {
        let others: Vec<usize> = (0..n).filter(|j| *j != i).collect();
        for size in 0..n {
            let coefficient = factorial(size) * factorial(n - size - 1) / factorial(n);
            for s in combos_of(&others, size) {
                let mut with = s.clone();
                with.push(i);
                let delta = game.wins(&with) as i32 as f64 - game.wins(&s) as i32 as f64;
                shapley[i] += coefficient * delta;
            }
        }
    }
    // Banzhaf : le nombre de coalitions adverses où le joueur est pivot.
    let swings = swing_counts(game);
    let absolute: Vec<f64> = swings
        .iter()
        .map(|c| *c as f64 / (2f64).powi((n - 1) as i32))
        .collect();
    let total: f64 = swings.iter().map(|c| *c as f64).sum();
    let normalise: Vec<f64> = swings.iter().map(|c| *c as f64 / total).collect();
    // Deegan-Packel : moyenne des 1/|S| sur les coalitions minimales.
    let minimal = game.minimal_winning();
    let deegan: Vec<f64> = (0..n)
        .map(|i| {
            let somme: f64 = minimal
                .iter()
                .filter(|s| s.contains(&i))
                .map(|s| 1.0 / s.len() as f64)
                .sum();
            somme / minimal.len() as f64
        })
        .collect();
    (shapley, absolute, normalise, deegan)
}

let game = game_exemple();
let (shapley, absolu, normalise, deegan) = power_indices(&game);
let poids_total: i64 = game.weights.iter().sum();
for i in 0..game.n() {
    println!(
        "{}  poids {}  part_poids {:.4}  shapley_shubik {:.4}  banzhaf_absolu {:.4}  banzhaf_normalise {:.4}  deegan_packel {:.4}",
        game.names[i],
        game.weights[i],
        game.weights[i] as f64 / poids_total as f64,
        shapley[i],
        absolu[i],
        normalise[i],
        deegan[i]
    );
}
let somme_shapley: f64 = shapley.iter().sum();
let somme_absolu: f64 = absolu.iter().sum();
let somme_normalise: f64 = normalise.iter().sum();
let somme_deegan: f64 = deegan.iter().sum();
println!(
    "Sommes : shapley_shubik {:.4}, banzhaf_absolu {:.4}, banzhaf_normalise {:.4}, deegan_packel {:.4}",
    somme_shapley, somme_absolu, somme_normalise, somme_deegan
);
let _ = &game;

A  poids 3  part_poids 0.5000  shapley_shubik 0.6667  banzhaf_absolu 0.7500  banzhaf_normalise 0.6000  deegan_packel 0.5000


B  poids 2  part_poids 0.3333  shapley_shubik 0.1667  banzhaf_absolu 0.2500  banzhaf_normalise 0.2000  deegan_packel 0.2500


C  poids 1  part_poids 0.1667  shapley_shubik 0.1667  banzhaf_absolu 0.2500  banzhaf_normalise 0.2000  deegan_packel 0.2500


Sommes : shapley_shubik 1.0000, banzhaf_absolu 1.2500, banzhaf_normalise 1.0000, deegan_packel 1.0000


## 2. Une seconde voie : énumérer des modèles SMT

La voie combinatoire énumère les coalitions à la main. La voie suivante confie au solveur les variables booléennes d'appartenance et la contrainte de pivot ou de minimalité. Après chaque modèle, une **clause de blocage** interdit exactement cette affectation : le solveur passe alors au modèle suivant, jusqu'à l'insatisfaisabilité.

Les deux voies institutionalisent la même idée — décider qu'une coalition gagne par une formule — mais aucune ne dépend de l'autre : l'énumération combinatoire et la recherche de modèles Z3 ne partagent que la définition mathématique du jeu, pas le mécanisme de recherche. C'est ce qui fait du test différentiel suivant une mesure et non une tautologie.

In [5]:
use z3::ast::{Ast, Bool, Int};
use z3::{Solver, SatResult};

// Somme d'expressions entières.
fn int_sum(termes: &[Int]) -> Int {
    let mut acc = Int::from_i64(0);
    for t in termes {
        acc = Int::add(&[acc, t.clone()]);
    }
    acc
}

// Les coalitions minimales gagnantes, trouvées par le solveur : variables
// booléennes d'appartenance, contrainte de poids, minimalité, puis clause de
// blocage qui interdit exactement l'affectation trouvée.
fn minimal_winning_smt(game: &VotingGame) -> Vec<Vec<usize>> {
    let x: Vec<Bool> = (0..game.n())
        .map(|i| Bool::new_const(format!("m_{}", i)))
        .collect();
    let zero = Int::from_i64(0);
    let termes: Vec<Int> = (0..game.n())
        .map(|i| x[i].ite(&Int::from_i64(game.weights[i]), &zero))
        .collect();
    let poids = int_sum(&termes);
    let mut solver = Solver::new();
    solver.assert(&poids.ge(&Int::from_i64(game.quota)));
    for i in 0..game.n() {
        let sans_i = Int::add(&[poids.clone(), Int::from_i64(-game.weights[i])]);
        solver.assert(&Bool::or(&[
            &x[i].not(),
            &sans_i.lt(&Int::from_i64(game.quota)),
        ]));
    }
    let mut trouvees: Vec<Vec<usize>> = Vec::new();
    loop {
        if solver.check() != SatResult::Sat {
            break;
        }
        let model = match solver.get_model() {
            Some(m) => m,
            None => break,
        };
        let mut affectation: Vec<bool> = Vec::new();
        for i in 0..game.n() {
            let valeur = model
                .eval(&x[i], true)
                .and_then(|b| b.as_bool())
                .unwrap_or(false);
            affectation.push(valeur);
        }
        trouvees.push(
            affectation
                .iter()
                .enumerate()
                .filter(|(_, present)| **present)
                .map(|(i, _)| i)
                .collect(),
        );
        // Clause de blocage : interdire exactement cette affectation.
        let blocage: Vec<Bool> = (0..game.n())
            .map(|i| x[i].ne(Bool::from_bool(affectation[i])))
            .collect();
        solver.assert(&Bool::or(&blocage));
    }
    trouvees
}

// Les tailles de coalition où le joueur est pivot, comptées par le solveur.
fn swing_sizes_smt(game: &VotingGame, joueur: usize) -> Vec<(usize, i64)> {
    let autres: Vec<usize> = (0..game.n()).filter(|j| *j != joueur).collect();
    let x: Vec<Bool> = autres
        .iter()
        .map(|j| Bool::new_const(format!("s_{}_{}", joueur, j)))
        .collect();
    let zero = Int::from_i64(0);
    let termes: Vec<Int> = (0..autres.len())
        .map(|k| x[k].ite(&Int::from_i64(game.weights[autres[k]]), &zero))
        .collect();
    let poids = int_sum(&termes);
    let mut solver = Solver::new();
    solver.assert(&poids.lt(&Int::from_i64(game.quota)));
    solver.assert(
        &Int::add(&[poids.clone(), Int::from_i64(game.weights[joueur])])
            .ge(&Int::from_i64(game.quota)),
    );
    let mut comptes: Vec<(usize, i64)> = Vec::new();
    loop {
        if solver.check() != SatResult::Sat {
            break;
        }
        let model = match solver.get_model() {
            Some(m) => m,
            None => break,
        };
        let mut affectation: Vec<bool> = Vec::new();
        for k in 0..autres.len() {
            let valeur = model
                .eval(&x[k], true)
                .and_then(|b| b.as_bool())
                .unwrap_or(false);
            affectation.push(valeur);
        }
        let taille = affectation.iter().filter(|present| **present).count();
        match comptes.iter_mut().find(|(t, _)| *t == taille) {
            Some(entree) => entree.1 += 1,
            None => comptes.push((taille, 1)),
        }
        let blocage: Vec<Bool> = (0..autres.len())
            .map(|k| x[k].ne(Bool::from_bool(affectation[k])))
            .collect();
        solver.assert(&Bool::or(&blocage));
    }
    comptes
}

// Les indices par la voie SMT : Shapley-Shubik, Banzhaf normalisé et
// Deegan-Packel.
fn power_indices_smt(game: &VotingGame) -> (Vec<f64>, Vec<f64>, Vec<f64>, Vec<Vec<usize>>) {
    let n = game.n();
    let mut shapley = vec![0.0f64; n];
    let mut pivots = vec![0i64; n];
    for i in 0..n {
        for (taille, compte) in swing_sizes_smt(game, i) {
            let coefficient = factorial(taille) * factorial(n - taille - 1) / factorial(n);
            shapley[i] += coefficient * compte as f64;
            pivots[i] += compte;
        }
    }
    let total: f64 = pivots.iter().map(|c| *c as f64).sum();
    let normalise: Vec<f64> = pivots.iter().map(|c| *c as f64 / total).collect();
    let minimales = minimal_winning_smt(game);
    let deegan: Vec<f64> = (0..n)
        .map(|i| {
            let somme: f64 = minimales
                .iter()
                .filter(|s| s.contains(&i))
                .map(|s| 1.0 / s.len() as f64)
                .sum();
            somme / minimales.len() as f64
        })
        .collect();
    (shapley, normalise, deegan, minimales)
}

let game = game_exemple();
let (shapley_smt, normalise_smt, deegan_smt, minimales_smt) = power_indices_smt(&game);
for i in 0..game.n() {
    println!(
        "{}  shapley_shubik {:.4}  banzhaf_normalise {:.4}  deegan_packel {:.4}",
        game.names[i],
        shapley_smt[i],
        normalise_smt[i],
        deegan_smt[i]
    );
}
let coalitions_minimales: Vec<String> = minimales_smt
    .iter()
    .map(|s| {
        let noms: Vec<String> = s.iter().map(|i| format!("'{}'", game.names[*i])).collect();
        format!("[{}]", noms.join(", "))
    })
    .collect();
println!("Modèles minimaux Z3 : [{}]", coalitions_minimales.join(", "));
let _ = &game;

A  shapley_shubik 0.6667  banzhaf_normalise 0.6000  deegan_packel 0.5000


B  shapley_shubik 0.1667  banzhaf_normalise 0.2000  deegan_packel 0.2500


C  shapley_shubik 0.1667  banzhaf_normalise 0.2000  deegan_packel 0.2500


Modèles minimaux Z3 : [['A', 'B'], ['A', 'C']]


### Cross-validation des deux voies

Comparer une seule instance peut masquer une erreur symétrique. Nous croisons donc trois structures : l'exemple du début, un jeu symétrique (`[2; 1, 1, 1]`, où les trois joueurs sont interchangeables) et un jeu à quatre acteurs. Pour chacune, la voie combinatoire et la voie SMT calculent les mêmes trois indices, et l'on mesure l'écart maximal.

L'écart attendu est nul à la précision flottante près. C'est un **test différentiel** sur neuf couples jeu/indice, pas une preuve universelle : l'espace exploré reste fini et choisi.

In [6]:
// Les trois jeux de cross-validation : exemple, symétrique, quatre acteurs.
let jeux = [game_exemple(), game_symetrique(), game_quatre()];
let mut ecart_max_global = 0.0f64;
println!("{:>3} {:>18} {:>13}", "jeu", "indice", "ecart_max");
for (rang, game) in jeux.iter().enumerate() {
    let (shapley, _absolu, normalise, deegan) = power_indices(game);
    let (shapley_smt, normalise_smt, deegan_smt, _minimales) = power_indices_smt(game);
    let ecarts: Vec<(&str, f64)> = vec![
        (
            "shapley_shubik",
            (0..game.n())
                .map(|i| (shapley[i] - shapley_smt[i]).abs())
                .fold(0.0, f64::max),
        ),
        (
            "banzhaf_normalise",
            (0..game.n())
                .map(|i| (normalise[i] - normalise_smt[i]).abs())
                .fold(0.0, f64::max),
        ),
        (
            "deegan_packel",
            (0..game.n())
                .map(|i| (deegan[i] - deegan_smt[i]).abs())
                .fold(0.0, f64::max),
        ),
    ];
    for (indice, ecart) in ecarts {
        ecart_max_global = ecart_max_global.max(ecart);
        println!("{:>3} {:>18} {:.6e}", rang + 1, indice, ecart);
    }
}
if ecart_max_global < 1e-12 {
    println!("Cross-validation réussie : 3 jeux, 3 indices, écart maximal < 1e-12");
} else {
    println!("Cross-validation échouée : écart maximal {:e}", ecart_max_global);
}
let _ = &jeux;

jeu             indice     ecart_max


  1     shapley_shubik 0.000000e0


  1  banzhaf_normalise 0.000000e0


  1      deegan_packel 0.000000e0


  2     shapley_shubik 0.000000e0


  2  banzhaf_normalise 0.000000e0


  2      deegan_packel 0.000000e0


  3     shapley_shubik 5.551115e-17


  3  banzhaf_normalise 0.000000e0


  3      deegan_packel 0.000000e0


Cross-validation réussie : 3 jeux, 3 indices, écart maximal < 1e-12


### Exemple guidé — un jeu de vote à joueur nul

**Énoncé.** Le jeu `[10; 5, 3, 2, 1]` compte quatre joueurs : D, avec un poids de 1, ne fait jamais basculer une décision. Calculer les indices par les deux voies et le vérifier.

**Résolution.** D est un joueur **nul** : aucune coalition où il entre ne change le résultat de celle où il manque. Ses trois indices valent donc exactement 0. La coalition minimale gagnante est `{A, B, C}` : 5 + 3 + 2 = 10 atteint le quota, et toute coalition de poids strictement inférieur échoue.

C'est le contre-exemple exact du jeu `[4; 3, 2, 1]` du début, où C avait un poids de 16,7 % et un pouvoir non nul : **le poids ne suffit pas à résumer le pouvoir**.

In [7]:
// Exemple guidé résolu : le jeu [10; 5, 3, 2, 1], D est le joueur nul.
// `zero_propre` évite d'afficher « -0.0000 » pour une grandeur nulle.
fn zero_propre(x: f64) -> f64 {
    if x == 0.0 {
        0.0
    } else {
        x
    }
}

let game = game_joueur_nul();
let (shapley, _absolu, normalise, deegan) = power_indices(&game);
let (shapley_smt, normalise_smt, deegan_smt, minimales_smt) = power_indices_smt(&game);
println!("Indices par la voie combinatoire :");
for i in 0..game.n() {
    println!(
        "{}  shapley_shubik {:.4}  banzhaf_normalise {:.4}  deegan_packel {:.4}",
        game.names[i],
        zero_propre(shapley[i]),
        zero_propre(normalise[i]),
        zero_propre(deegan[i])
    );
}
println!("Indices par la voie SMT :");
for i in 0..game.n() {
    println!(
        "{}  shapley_shubik {:.4}  banzhaf_normalise {:.4}  deegan_packel {:.4}",
        game.names[i],
        zero_propre(shapley_smt[i]),
        zero_propre(normalise_smt[i]),
        zero_propre(deegan_smt[i])
    );
}
let coalitions_minimales: Vec<String> = minimales_smt
    .iter()
    .map(|s| {
        let noms: Vec<String> = s.iter().map(|i| format!("'{}'", game.names[*i])).collect();
        format!("[{}]", noms.join(", "))
    })
    .collect();
println!(
    "Coalitions minimales gagnantes (SMT) : [{}]",
    coalitions_minimales.join(", ")
);
let d = 3;
println!(
    "Joueur nul D : Shapley-Shubik = {:.4} et Banzhaf normalisé = {:.4} dans les deux voies.",
    zero_propre(shapley[d]),
    zero_propre(normalise[d])
);
let ecart = (0..game.n())
    .map(|i| (shapley[i] - shapley_smt[i]).abs())
    .fold(0.0, f64::max);
println!(
    "Concordance des deux voies : écart maximal {:.2e} sur les trois indices.",
    ecart
);
let _ = &game;

Indices par la voie combinatoire :


A  shapley_shubik 0.3333  banzhaf_normalise 0.3333  deegan_packel 0.3333


B  shapley_shubik 0.3333  banzhaf_normalise 0.3333  deegan_packel 0.3333


C  shapley_shubik 0.3333  banzhaf_normalise 0.3333  deegan_packel 0.3333


D  shapley_shubik 0.0000  banzhaf_normalise 0.0000  deegan_packel 0.0000


Indices par la voie SMT :


... (3 outputs left out)

D  shapley_shubik 0.0000  banzhaf_normalise 0.0000  deegan_packel 0.0000


Coalitions minimales gagnantes (SMT) : [['A', 'B', 'C']]


Joueur nul D : Shapley-Shubik = 0.0000 et Banzhaf normalisé = 0.0000 dans les deux voies.


Concordance des deux voies : écart maximal 0.00e0 sur les trois indices.


### Exercice 1 — Diagnostiquer les rôles structurels

**Objectif** : compléter une fonction qui retourne les joueurs nuls, à veto et dictateurs d'un jeu.

**Indice** : utilisez `is_dummy`, `is_veto` et `is_dictator`.

**Étapes.**

1. Parcourez les joueurs du jeu.
2. Pour chacun, appelez les trois prédicats structurels.
3. Rassemblez les noms dans trois vecteurs : nuls, à veto, dictateurs.

Le gabarit renvoie trois vecteurs vides : les exercices non complétés restent des activités proposées au lecteur, et le carnet s'exécute de bout en bout.

In [8]:
// Exercice 1 : diagnostiquer les rôles structurels d'un jeu.
// TODO étudiant : renvoyez les joueurs nuls, les joueurs à veto et les
// dictateurs, sous forme de trois vecteurs de noms.
// Étape 1 : parcourez les joueurs du jeu.
// Étape 2 : appelez is_dummy, is_veto et is_dictator pour chacun.
fn diagnostiquer_roles(game: &VotingGame) -> (Vec<String>, Vec<String>, Vec<String>) {
    let _ = game;
    (Vec::new(), Vec::new(), Vec::new())
}

let (nuls, vetos, dictateurs) = diagnostiquer_roles(&game_exemple());
println!(
    "Exercice 1 à compléter : joueurs nuls = {:?}, veto = {:?}, dictateurs = {:?}",
    nuls, vetos, dictateurs
);

Exercice 1 à compléter : joueurs nuls = [], veto = [], dictateurs = []


### Exercice 2 — Un dictateur absorbe tout le pouvoir

**Objectif** : construire un jeu à quatre joueurs dont l'un est un **dictateur** — il gagne seul, sans aucun allié — puis vérifier que tous les autres joueurs ont un pouvoir nul et que les deux voies s'accordent.

**Étapes.**

1. Construisez le jeu : un joueur dont le poids atteint le quota, les autres de poids plus faibles, avec un quota que seul le dictateur atteint.
2. Calculez les indices par la voie combinatoire.
3. Calculez les indices par la voie SMT.
4. Vérifiez que le dictateur vaut 1 et que les autres valent 0 dans les deux tables.

**Indice** : un dictateur doit satisfaire `is_dictator` : il gagne seul **et** il a le veto.

In [9]:
// Exercice 2 : construire un jeu à dictateur et vérifier les deux voies.
// TODO étudiant :
// Étape 1 : construisez VotingGame::new(vec![...], quota, vec![...]).
// Étape 2 : calculez les indices par la voie combinatoire (power_indices).
// Étape 3 : calculez les indices par la voie SMT (power_indices_smt).
// Étape 4 : vérifiez que le dictateur vaut 1 et que les autres valent 0.
let jeu_dictateur: Option<VotingGame> = None; // TODO étudiant : le jeu à dictateur
match &jeu_dictateur {
    Some(g) => {
        let (shapley, _absolu, normalise, _deegan) = power_indices(g);
        let (shapley_smt, normalise_smt, _deegan_smt, _minimales) = power_indices_smt(g);
        let autres_nuls = (1..g.n()).all(|i| {
            shapley[i].abs() < 1e-12
                && normalise[i].abs() < 1e-12
                && shapley_smt[i].abs() < 1e-12
                && normalise_smt[i].abs() < 1e-12
        });
        println!(
            "Dictateur : shapley {:.4} (SMT {:.4}) ; les autres joueurs sont nuls : {}",
            shapley[0], shapley_smt[0], autres_nuls
        );
    }
    None => println!("Exercice 2 à compléter : construire un jeu à dictateur"),
}
let _ = &jeu_dictateur;

Exercice 2 à compléter : construire un jeu à dictateur


### Exercice 3 — Tester la symétrie bornée

**Objectif** : ajouter à un solveur deux joueurs interchangeables, puis chercher un modèle où leurs valeurs de Shapley-Shubik diffèrent.

**Indice** : pour chaque coalition ne contenant ni le joueur 0 ni le joueur 1, contraindre `v(S ∪ {0}) == v(S ∪ {1})`, puis ajouter l'inégalité `v(0) != v(1)` et appeler `check()`. Sur un jeu simple et monotone borné, le statut attendu est `unsat` — et c'est cette borne qu'il faut énoncer honnêtement, pas une généralisation sans limite.

In [10]:
// Exercice 3 : symétrie bornée — le statut UNSAT attendu.
// TODO étudiant :
// Étape 1 : reconstruisez les variables de coalition (valeurs symboliques).
// Étape 2 : imposez l'interchangeabilité des joueurs 0 et 1.
// Étape 3 : ajoutez la négation de l'égalité de leurs indices.
// Étape 4 : appelez check() et renvoyez le statut.
fn statut_symetrie_bornee() -> Option<String> {
    // TODO étudiant : le solveur et son statut ("unsat" attendu).
    None
}

let statut = statut_symetrie_bornee();
println!(
    "Exercice 3 à compléter : statut attendu UNSAT (statut courant : {:?})",
    statut
);

Exercice 3 à compléter : statut attendu UNSAT (statut courant : None)


## 3. Étude de cas : une photographie structurelle

Le jumeau Python étudie l'Assemblée nationale à l'automne 2024 : douze groupes, 577 sièges, un quota de 289. Chaque groupe est traité comme un acteur unique, avec une discipline de vote supposée parfaite.

La voie combinatoire suffit ici : l'énumération porte sur 2¹² = 4 096 coalitions, un volume que le portage Rust traite sans peine. La voie SMT, elle, demanderait 12 × 2¹¹ énumérations de modèles — hors de portée d'un carnet pédagogique, et c'est pourquoi le jumeau Python s'en tient lui aussi à la voie combinatoire sur cette instance.

In [11]:
// L'Assemblée à l'automne 2024 : douze groupes, 577 sièges, quota 289.
let groupes: Vec<(&str, i64)> = vec![
    ("RN", 126),
    ("EPR", 99),
    ("LFI", 72),
    ("SOC", 66),
    ("DR", 47),
    ("EcoS", 38),
    ("DEM", 36),
    ("HOR", 31),
    ("LIOT", 23),
    ("GDR", 17),
    ("UDR", 16),
    ("NI", 6),
];
let assemblee = VotingGame::new(
    groupes.iter().map(|(_, w)| *w).collect(),
    289,
    groupes.iter().map(|(n, _)| *n).collect(),
);
let (shapley, absolu, normalise, deegan) = power_indices(&assemblee);
let sieges: i64 = groupes.iter().map(|(_, w)| *w).sum();
for i in 0..assemblee.n() {
    println!(
        "{}  poids {}  part_poids {:.4}  shapley_shubik {:.4}  banzhaf_absolu {:.4}  banzhaf_normalise {:.4}  deegan_packel {:.4}",
        assemblee.names[i],
        assemblee.weights[i],
        assemblee.weights[i] as f64 / sieges as f64,
        shapley[i],
        absolu[i],
        normalise[i],
        deegan[i]
    );
}
let minimales = assemblee.minimal_winning();
println!(
    "Contrôle des sièges : {} ; coalitions minimales : {}",
    sieges,
    minimales.len()
);
let _ = &assemblee;

RN  poids 126  part_poids 0.2184  shapley_shubik 0.2459  banzhaf_absolu 0.5361  banzhaf_normalise 0.2408  deegan_packel 0.1020


EPR  poids 99  part_poids 0.1716  shapley_shubik 0.1811  banzhaf_absolu 0.3838  banzhaf_normalise 0.1724  deegan_packel 0.0957


LFI  poids 72  part_poids 0.1248  shapley_shubik 0.1235  banzhaf_absolu 0.2744  banzhaf_normalise 0.1232  deegan_packel 0.0957


SOC  poids 66  part_poids 0.1144  shapley_shubik 0.1126  banzhaf_absolu 0.2471  banzhaf_normalise 0.1110  deegan_packel 0.0910


DR  poids 47  part_poids 0.0815  shapley_shubik 0.0761  banzhaf_absolu 0.1748  banzhaf_normalise 0.0785  deegan_packel 0.0965


EcoS  poids 38  part_poids 0.0659  shapley_shubik 0.0610  banzhaf_absolu 0.1416  banzhaf_normalise 0.0636  deegan_packel 0.0890


... (3 outputs left out)

GDR  poids 17  part_poids 0.0295  shapley_shubik 0.0268  banzhaf_absolu 0.0635  banzhaf_normalise 0.0285  deegan_packel 0.0738


UDR  poids 16  part_poids 0.0277  shapley_shubik 0.0237  banzhaf_absolu 0.0576  banzhaf_normalise 0.0259  deegan_packel 0.0708


NI  poids 6  part_poids 0.0104  shapley_shubik 0.0084  banzhaf_absolu 0.0205  banzhaf_normalise 0.0092  deegan_packel 0.0358


Contrôle des sièges : 577 ; coalitions minimales : 184


## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| Le poids ne suffit pas à résumer le pouvoir | dans le jeu `VotingGame(weights=(3, 2, 1), quota=4, names=('A','B','C'))`, A porte `0.5000` du poids mais `0.6667` au Shapley-Shubik et `0.7500` au Banzhaf absolu |
| Les coalitions minimales suffisent à caractériser la structure | `{A,B}` et `{A,C}` sont les deux coalitions minimales de ce jeu, et A figure dans les deux |
| Un poids non nul peut donner un pouvoir nul | D pèse 1 sur 11 dans le jeu `[10; 5,3,2,1]` et ses trois indices valent `0.0000` |
| Deux voies indépendantes valent mieux qu'une | l'écart maximal entre la voie combinatoire et la voie SMT tombe à l'ordre de la précision flottante sur 9 couples jeu/indice |

Ce que ce carnet **ne** démontre pas : l'égalité des deux voies n'est vérifiée que sur les instances affichées. Un test différentiel sur un ensemble fini et choisi n'est pas une preuve générale — c'est un garde-fou contre l'erreur d'implémentation, pas un théorème. La généralisation reste l'objet du jumeau Lean de la série.

## Pour aller plus loin

- [Jumeau Python](Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) : la même institution, avec l'encodage SMT en Python et la matrice d'axiomes.
- [Index de la sous-série Coalitions](README.md) : le fil de la sous-série et ce que chaque carnet y ajoute.
- [Index de la série GameTheory](../README.md) : la place du pouvoir coalitionnel parmi les autres structures de la théorie des jeux.

## Références

* Shapley, L. S. (1953). *A Value for n-Person Games*. In: Kuhn, H. W. & Tucker, A. W. (Eds.), Contributions to the Theory of Games, vol. II (Annals of Mathematics Studies 28), Princeton University Press, pp. 307-317. -> §1 (le geste fondateur : la valeur de Shapley `phi_i(v)` définit le pouvoir d'un joueur comme la contribution marginale moyenne sur toutes les coalitions).

* Shapley, L. S. & Shubik, M. (1954). *A Method for Evaluating the Distribution of Power in a Committee System*. American Political Science Review 48(3): 787-792. -> §1 (l'indice de Shapley-Shubik, spécialisée au cas des jeux de vote simples) et §3 (l'étude politique d'une photographie structurelle d'un scrutin).

* Banzhaf, J. F. III (1965). *Weighted Voting Doesn't Work: A Mathematical Analysis*. Rutgers Law Review 19(2): 317-343. -> §1 (l'indice de Banzhaf, pouvoir absolu normalisé).

* Penrose, L. S. (1946). *The Elementary Statistics of Majority Voting*. Journal of the Royal Statistical Society 109(1): 53-57. -> §1 (le cas limite des grands comités).

* Riker, W. H. (1962). *The Theory of Political Coalitions*. Yale University Press. -> §3 (l'analyse des coalitions politiques comme équilibres de pouvoir).

* Felsenthal, D. S. & Machover, M. (1998). *The Measurement of Voting Power: Theory and Practice, Problems and Paradoxes*. Edward Elgar, Cheltenham. -> §3 (la mesure empirique du pouvoir dans les institutions réelles).

* Z3 — solveur SMT : <https://github.com/Z3Prover/z3> ; liaison Rust : crate [`z3`](https://crates.io/crates/z3) (le crate est lié dynamiquement à la bibliothèque Z3 installée sur le poste ; la version de la bibliothèque relevée par le carnet est affichée dans la première cellule).